# 05 — End-to-End Redaction Pipeline

```
input_folder/*.pdf
  → temp_images/{stem}_page_{n}.png          (PyMuPDF render)
  → redacted_text/{stem}_page_{n}.json       (Bedrock: sanitized text + mapping)
  → output_folder/redacted_{stem}.pdf        (ReportLab: sanitized content)
  → output_folder/summary_{stem}.pdf         (ReportLab: replacement mapping table)
  → output_folder/governance_{stem}.json     (Governance: per-page redaction log)
  → cleanup temp_images/ + redacted_text/
```

PII/PHI is **replaced with realistic fictitious dummy values** — consistent across all pages.
Each source document produces three output files: `redacted_{stem}.pdf` (sanitized content), `summary_{stem}.pdf` (replacement mapping table), and `governance_{stem}.json` (machine-readable audit log).

**Parallelism:** Multiple PDFs are processed concurrently via `ThreadPoolExecutor` (default 3 workers). Pages within each PDF remain sequential for cross-page mapping consistency. Tune `MAX_WORKERS` in the Configuration cell based on your Bedrock TPS limit.

Edit the **Configuration** cell, then **Run All**.

In [ ]:
%pip install pymupdf boto3 Pillow reportlab python-dotenv --prefer-binary -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────
import os, sys
from pathlib import Path

BASE_DIR      = Path(os.getcwd()).parent
# Add project root to path so we can import from models/
sys.path.insert(0, str(BASE_DIR))

INPUT_FOLDER  = BASE_DIR / "input_folder"
OUTPUT_FOLDER = BASE_DIR / "output_folder"
TEMP_FOLDER   = BASE_DIR / "temp_images"
TEXT_FOLDER   = BASE_DIR / "redacted_text"

# Choose model: change this to any key from config/models.json
# e.g. "sonnet 3.7", "haiku 4.5", "sonnet 4.5", "opus 4.6"
SELECTED_MODEL = None  # None = use default from config/models.json

PAGE_DPI      = 230
MAX_TOKENS    = 4096
RETRY_LIMIT   = 3
RETRY_DELAY   = 5
CLEAN_UP      = True
MAX_WORKERS   = 3     # concurrent PDFs — tune up if your Bedrock TPS limit allows

for folder in [OUTPUT_FOLDER, TEMP_FOLDER, TEXT_FOLDER]:
    folder.mkdir(exist_ok=True)

print("Configuration OK")

In [ ]:
# ── Imports & logger ───────────────────────────────────────────
import fitz, json, base64, time, threading, logging
from collections import defaultdict
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER

from utils import (get_logger, extract_json, validate_mapping,
                   fix_remaining_violations, enforce_replacements_in_text,
                   audit_sanitized_text, check_duplicate_replacements,
                   fix_duplicate_replacements)
from models import get_bedrock_client, resolve_model_id, get_audit_model_id
logger = get_logger("05_pipeline")

BEDROCK_MODEL = resolve_model_id(SELECTED_MODEL)
AUDIT_MODEL = get_audit_model_id()
bedrock = get_bedrock_client()

_LOG_DIR = BASE_DIR / "logs"
_LOG_DIR.mkdir(exist_ok=True)
_LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(message)s"
_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"


def get_file_logger(stem: str) -> logging.Logger:
    """Create a per-document logger that writes to logs/{stem}.log."""
    name = f"doc_{stem}"
    log = logging.getLogger(name)
    if log.handlers:
        return log  # already set up (e.g. retry of same file)
    log.setLevel(logging.DEBUG)
    fh = logging.FileHandler(str(_LOG_DIR / f"{stem}.log"), mode="w", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter(_LOG_FORMAT, datefmt=_DATE_FORMAT))
    log.addHandler(fh)
    # Also write to the shared pipeline console so notebook still shows progress
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter(_LOG_FORMAT, datefmt=_DATE_FORMAT))
    log.addHandler(ch)
    log.propagate = False
    return log


logger.info("Imports OK — model: %s", BEDROCK_MODEL)

In [ ]:
# ── Prompt + helper functions ──────────────────────────────────

_BASE_PROMPT = """You are a data-sanitization assistant. For each page image, transcribe all \
text and replace every PII/PHI value with a completely different fictitious value that shares \
NO words with the original.

  Correct:  "MARGARET HOLMES" → @DIANA CHEN@ in text, "Diana Chen" in mapping
  Wrong:    "MARGARET HOLMES" → "MARGARET CHEN"  ← shares a word, not allowed
  Wrong:    "MARGARET HOLMES" → "MARGARET HOLMES" ← echoed the original, not allowed

Redact and replace ONLY these PII/PHI categories:
- Full names in any format ("First Last", "Last, First", with titles/suffixes/initials), \
including names after role labels such as "Claimant:", "Patient:", "Provider:", "Insured:", \
"Applicant:", "Member:", "Beneficiary:", etc.
- Email addresses
- Phone and fax numbers
- Social Security Numbers (SSN) or national identifiers
- Dates of Birth (DOB only — not date of service, date of injury, or other dates)
- Medical record numbers (MRN, patient ID, chart number, etc.)
- Medical diagnoses or conditions tied to individuals (ICD codes, clinical descriptions)
- Credit card details (card numbers, expiration dates, CVVs, cardholder names)

Leave unchanged:
- Physical mailing addresses
- Insurance, policy, group, or claim numbers
- Non-DOB dates (date of service, injury, admission/discharge, etc.)
- Driver's license or state ID numbers
- Bank account or routing numbers
- Employer names or job titles
- Facility, hospital, or clinic names
- Organization, company, church, or entity names (e.g. "Pacific Ridge Casualty Company", \
"Diocese of Brooklyn", "First National Bank", "Blue Cross Blue Shield"). \
These are NOT personal names even if they appear after labels like "Insured:" or "Provider:".

Rules:
1. Transcribe ALL text — headings, labels, table cells, footers, and all form elements.
2. For checkboxes and form fields: reproduce the checked/unchecked state faithfully. \
Use [X] for checked boxes and [ ] for unchecked boxes. If a form has radio buttons or \
checkmarks (✓, ☑), transcribe them as [X]; empty boxes (☐, ○) as [ ]. \
Preserve which option is selected — do NOT leave all boxes unchecked.
3. For multi-column layouts (e.g. a form with a left section and right section): \
transcribe each column or section completely before moving to the next. \
Use a clear separator like "--- LEFT COLUMN ---" and "--- RIGHT COLUMN ---" or \
"[Section: Left]" / "[Section: Right]" to mark where each column starts. \
Do NOT interleave lines from different columns — finish one column fully, then start the next.
4. Replace every PII/PHI value inline. The replacement must be a completely different \
invented value — never echo the original. Use the same replacement whenever the same \
original appears.
5. **@ delimiters**: When writing replacement values into the transcribed text, wrap them \
in @ delimiters. Example: if the replacement for "John Smith" is "Alex Rivera", write \
@Alex Rivera@ in the text. In the mapping JSON, write the replacement WITHOUT @ \
(just "Alex Rivera"). This applies to ALL replacement types (names, SSNs, DOBs, etc.).
6. **Unique replacements**: Every distinct original value MUST receive a unique replacement. \
Never assign the same replacement to two different people or values. If the document \
has three different people, they must get three completely different replacement names. \
Similarly, two different SSNs must get two different replacement SSNs.
7. Pay special attention to names inside table cells, form fields, and after role labels \
(e.g. "Claimant:", "Patient:"). These are PII regardless of formatting or context. \
However, do NOT redact organization, company, church, legal entity, or facility names — \
only redact names of individual people. "Pacific Ridge Casualty" is a company, not a person. \
"Diocese of Brooklyn" is an organization, not a person.
8. Replacements must look realistic but must not correspond to real people.
9. In the mapping, write only a partially masked hint of the original \
(e.g. "J*** S***" for "John Smith") — never the full original value. \
The "replacement" field must contain the NEW fictitious value WITHOUT @ delimiters.

<<PRIOR_MAPPING>>
Return ONLY valid JSON with no markdown fences:
{
  "sanitized_text": "Claimant: @Alex Rivera@\\nSSN: @456-78-9012@\\n...",
  "mapping": [
    {"original_masked": "J*** S***", "replacement": "Alex Rivera", "type": "Name"},
    {"original_masked": "***-**-6789", "replacement": "456-78-9012", "type": "SSN"}
  ]
}

Now transcribe and sanitize the following document page:"""

_BEDROCK_IMAGE_LIMIT = 5 * 1024 * 1024  # 5 MB — Bedrock limit on base64 string length


def _image_to_base64(image_path: Path, log=None) -> tuple[str, str]:
    _log = log or logger
    from PIL import Image
    import io

    raw = image_path.read_bytes()
    encoded = base64.standard_b64encode(raw)
    if len(encoded) <= _BEDROCK_IMAGE_LIMIT:
        return encoded.decode(), "image/png"

    img = Image.open(image_path).convert("RGB")
    for quality in (85, 70, 55, 40):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=quality, optimize=True)
        data = buf.getvalue()
        encoded = base64.standard_b64encode(data)
        if len(encoded) <= _BEDROCK_IMAGE_LIMIT:
            _log.debug("Compressed %s to %.1f MB (quality=%d)",
                       image_path.name, len(data) / 1024 / 1024, quality)
            return encoded.decode(), "image/jpeg"

    img = img.resize((img.width // 2, img.height // 2), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=70)
    data = buf.getvalue()
    _log.warning("Scaled down %s to %.1f MB", image_path.name, len(data) / 1024 / 1024)
    return base64.standard_b64encode(data).decode(), "image/jpeg"


def build_prompt(prior_mapping: list[dict]) -> str:
    if not prior_mapping:
        return _BASE_PROMPT.replace("<<PRIOR_MAPPING>>\n", "")
    rows = "\n".join(
        f"  - {r['original_masked']} → {r['replacement']} ({r['type']})"
        for r in prior_mapping
    )
    section = (
        "Previously identified mappings from earlier pages — "
        "use these EXACT replacements (wrapped in @) for any matching values on this page. "
        "Do NOT reuse any of these replacement values for a different original:\n"
        + rows + "\n\n"
    )
    return _BASE_PROMPT.replace("<<PRIOR_MAPPING>>", section.rstrip())


def pdf_to_images(pdf_path: Path, log=None) -> list[Path]:
    _log = log or logger
    _log.info("Rendering pages: %s", pdf_path.name)
    doc  = fitz.open(str(pdf_path))
    zoom = PAGE_DPI / 72
    mat  = fitz.Matrix(zoom, zoom)
    paths = []
    for i in range(len(doc)):
        pix  = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
        dest = TEMP_FOLDER / f"{pdf_path.stem}_page_{i+1}.png"
        pix.save(str(dest))
        paths.append(dest)
        _log.debug("  Page %d → %s (%dx%d)", i+1, dest.name, pix.width, pix.height)
    doc.close()
    _log.info("%d page(s) rendered for %s", len(paths), pdf_path.name)
    return paths


def redact_image(image_path: Path, prior_mapping: list[dict],
                 bedrock_client=None, model_id=None, audit_model_id=None, log=None) -> dict:
    """Redact a single page image via Bedrock. Thread-safe when explicit client is passed."""
    _bedrock = bedrock_client or bedrock
    _model = model_id or BEDROCK_MODEL
    _audit = audit_model_id or AUDIT_MODEL
    _log = log or logger

    img_b64, media_type = _image_to_base64(image_path, log=_log)

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": media_type, "data": img_b64}},
                {"type": "text", "text": build_prompt(prior_mapping)}
            ]
        }]
    }

    last_exc = None
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            _log.debug("Bedrock call: %s (attempt %d)", image_path.name, attempt)
            resp = _bedrock.invoke_model(
                modelId=_model, body=json.dumps(body),
                contentType="application/json", accept="application/json"
            )
            raw = json.loads(resp["body"].read())["content"][0]["text"]
            _log.debug("Raw response (first 200 chars): %s", raw[:200].replace("\n", "\\n"))
            result = extract_json(raw)

            violations = validate_mapping(result)
            if violations:
                _log.warning(
                    "Mapping violations in %s: %d row(s) share words with originals — retrying",
                    image_path.name, len(violations)
                )
                try:
                    rows_text = "\n".join(
                        f"  - {r['original_masked']} → {r['replacement']} ({r['type']})"
                        for r in violations
                    )
                    retry_body = {
                        "anthropic_version": "bedrock-2023-05-31",
                        "max_tokens": 1024,
                        "messages": [{"role": "user", "content": [{"type": "text", "text": (
                            "These replacements share words with their originals and must be fixed. "
                            "For each row, produce a new replacement that shares NO words with the original. "
                            "Return ONLY valid JSON (no markdown fences):\n"
                            "{\"mapping\": [{\"original_masked\": \"...\", "
                            "\"replacement\": \"...\", \"type\": \"...\"}]}\n\n"
                            f"Rows to fix:\n{rows_text}"
                        )}]}]
                    }
                    resp2 = _bedrock.invoke_model(
                        modelId=_model, body=json.dumps(retry_body),
                        contentType="application/json", accept="application/json"
                    )
                    raw2 = json.loads(resp2["body"].read())["content"][0]["text"]
                    fixed = extract_json(raw2)
                    fix_map = {(r["original_masked"], r["type"]): r["replacement"]
                               for r in fixed.get("mapping", [])}
                    for row in result["mapping"]:
                        key = (row["original_masked"], row["type"])
                        if key in fix_map:
                            old_val = row["replacement"]
                            new_val = fix_map[key]
                            result["sanitized_text"] = result["sanitized_text"].replace(old_val, new_val)
                            row["replacement"] = new_val
                    remaining = validate_mapping(result)
                    if remaining:
                        _log.warning(
                            "%d violation(s) remain after retry in %s — applying synthetic fix",
                            len(remaining), image_path.name
                        )
                        fix_remaining_violations(result, remaining, _log)
                except Exception as exc2:
                    _log.error("Targeted retry failed for %s: %s", image_path.name, exc2)
                    fix_remaining_violations(result, violations, _log)

            enforce_replacements_in_text(result, _log)

            # Check and fix duplicate replacements
            dupes = check_duplicate_replacements(result)
            if dupes:
                _log.warning("Duplicate replacements in %s: %d row(s) — fixing",
                             image_path.name, len(dupes))
                fix_duplicate_replacements(result, dupes, _log)

            audit_sanitized_text(result, _bedrock, _audit, _log)

            return result

        except json.JSONDecodeError as exc:
            _log.warning(
                "Page %s: non-JSON response (image-only/banner page). "
                "Using raw text, empty mapping. Preview: %s",
                image_path.name, exc.doc[:200].replace("\n", "\\n")
            )
            return {"sanitized_text": exc.doc.strip(), "mapping": []}

        except Exception as exc:
            last_exc = exc
            _log.warning("Attempt %d/%d failed for %s: %s",
                         attempt, RETRY_LIMIT, image_path.name, exc)
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)

    _log.error("All retries exhausted for %s", image_path.name)
    raise last_exc


def merge_mapping(accumulated: list[dict], new_rows: list[dict], log=None) -> list[dict]:
    _log = log or logger
    seen = {(r["original_masked"], r["type"]) for r in accumulated}
    added = 0
    for row in new_rows:
        key = (row["original_masked"], row["type"])
        if key not in seen:
            accumulated.append(row)
            seen.add(key)
            added += 1
    if added:
        _log.debug("Mapping +%d entries (total %d)", added, len(accumulated))
    return accumulated


def xml_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def build_redacted_pdf(pdf_stem: str, page_texts: list[str], out_path: Path, log=None) -> None:
    _log = log or logger
    _log.info("Building redacted PDF: %s (%d pages)", out_path.name, len(page_texts))
    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    body_style = ParagraphStyle(
        "Body", parent=styles["Normal"],
        fontName="Courier", fontSize=9, leading=13, alignment=TA_LEFT,
    )
    story = []
    for idx, text in enumerate(page_texts):
        if idx > 0:
            story.append(PageBreak())
        _log.debug("  Page %d: %d chars", idx + 1, len(text))
        for line in text.splitlines():
            safe = xml_escape(line)
            story.append(Paragraph(safe, body_style) if safe.strip() else Spacer(1, 6))
    doc.build(story)
    _log.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


def build_summary_pdf(pdf_stem: str, mapping: list[dict], out_path: Path, log=None) -> None:
    _log = log or logger
    _log.info("Building summary PDF: %s (%d entries)", out_path.name, len(mapping))
    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    heading_style = ParagraphStyle(
        "Heading", parent=styles["Heading2"], alignment=TA_CENTER, spaceAfter=12,
    )
    story = [
        Paragraph(f"PII / PHI Replacement Summary — {pdf_stem}", heading_style),
        Spacer(1, 6),
    ]
    table_data = [["Original (masked)", "Replacement", "Data Type"]]
    for row in mapping:
        table_data.append([row.get("original_masked", ""),
                           row.get("replacement", ""),
                           row.get("type", "")])
    tbl = Table(table_data, colWidths=[6*cm, 7*cm, 4*cm], repeatRows=1)
    tbl.setStyle(TableStyle([
        ("BACKGROUND",    (0, 0), (-1, 0),  colors.HexColor("#2E4057")),
        ("TEXTCOLOR",     (0, 0), (-1, 0),  colors.white),
        ("FONTNAME",      (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("FONTSIZE",      (0, 0), (-1, -1), 8),
        ("ROWBACKGROUNDS",(0, 1), (-1, -1), [colors.white, colors.HexColor("#F2F2F2")]),
        ("GRID",          (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN",        (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING",   (0, 0), (-1, -1), 4),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 4),
        ("TOPPADDING",    (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]))
    story.append(tbl)
    doc.build(story)
    _log.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


def build_governance_json(pdf_stem: str, source_name: str, page_mappings: list,
                          model_id: str, audit_model_id: str, out_path: Path, log=None) -> None:
    _log = log or logger
    all_redactions = sum(len(pm) for pm in page_mappings)
    categories = sorted({r["type"] for pm in page_mappings for r in pm})

    pages = []
    for i, pm in enumerate(page_mappings, 1):
        pages.append({"page_number": i, "redactions": pm})

    seen = {}
    for i, pm in enumerate(page_mappings, 1):
        for r in pm:
            key = (r["original_masked"], r["type"])
            if key not in seen:
                seen[key] = {**r, "pages": [i]}
            elif i not in seen[key]["pages"]:
                seen[key]["pages"].append(i)

    gov = {
        "source_file": source_name,
        "processed_at": datetime.now(tz=timezone.utc).isoformat(timespec="seconds"),
        "model_id": model_id,
        "audit_model_id": audit_model_id,
        "total_pages": len(page_mappings),
        "total_redactions": all_redactions,
        "categories_found": categories,
        "pages": pages,
        "consolidated_mapping": list(seen.values()),
    }
    out_path.write_text(json.dumps(gov, ensure_ascii=False, indent=2), encoding="utf-8")
    _log.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


logger.info("Prompt + helper functions defined")

In [ ]:
# ── Main pipeline ──────────────────────────────────────────────
pdf_files = sorted(
    p for p in INPUT_FOLDER.iterdir()
    if p.is_file() and p.suffix.lower() == ".pdf"
)
total_files = len(pdf_files)
logger.info("Found %d PDF(s) in input_folder/", total_files)

if not pdf_files:
    raise FileNotFoundError("No PDFs found. Add files to input_folder/ and re-run.")

# ── Separate skipped vs to-process ────────────────────────────
to_process = []
skipped_files = 0
for pdf_path in pdf_files:
    redacted_path   = OUTPUT_FOLDER / f"redacted_{pdf_path.stem}.pdf"
    summary_path    = OUTPUT_FOLDER / f"summary_{pdf_path.stem}.pdf"
    if redacted_path.exists() and summary_path.exists():
        skipped_files += 1
        governance_path = OUTPUT_FOLDER / f"governance_{pdf_path.stem}.json"
        if not governance_path.exists():
            logger.info("SKIP (outputs exist, no governance — cache deleted): %s", pdf_path.name)
        else:
            logger.info("SKIP (all outputs exist): %s", pdf_path.name)
    else:
        to_process.append(pdf_path)

logger.info("%d to process, %d skipped", len(to_process), skipped_files)

# ── Per-document worker (runs in thread) ──────────────────────
_progress_lock = threading.Lock()
_completed = [0]


def process_pdf(pdf_path: Path) -> dict:
    """Process a single PDF end-to-end. Each document gets its own log file."""
    stem = pdf_path.stem
    log = get_file_logger(stem)
    file_start = time.time()

    log.info("START: %s", pdf_path.name)

    redacted_path   = OUTPUT_FOLDER / f"redacted_{stem}.pdf"
    summary_path    = OUTPUT_FOLDER / f"summary_{stem}.pdf"
    governance_path = OUTPUT_FOLDER / f"governance_{stem}.json"

    # Step 1 — PDF → images
    image_paths = pdf_to_images(pdf_path, log=log)
    total_pages = len(image_paths)

    # Step 2 — Redact via Bedrock (sequential within document for cross-page consistency)
    log.info("Redacting %d pages via Bedrock", total_pages)
    page_texts: list[str] = []
    page_mappings: list[list[dict]] = []
    accumulated_mapping: list[dict] = []

    for page_idx, img_path in enumerate(image_paths, 1):
        page_start = time.time()
        cache = TEXT_FOLDER / f"{img_path.stem}.json"
        if cache.exists():
            log.info("[cache] page %d/%d", page_idx, total_pages)
            data = json.loads(cache.read_text(encoding="utf-8"))
            violations = validate_mapping(data)
            if violations:
                log.warning("[cache] %d violation(s) page %d — fixing",
                            len(violations), page_idx)
                fix_remaining_violations(data, violations, log)
            enforce_replacements_in_text(data, log)
            audit_sanitized_text(data, bedrock, AUDIT_MODEL, log)
        else:
            log.info("Redacting page %d/%d", page_idx, total_pages)
            data = redact_image(img_path, prior_mapping=accumulated_mapping,
                                bedrock_client=bedrock, model_id=BEDROCK_MODEL,
                                audit_model_id=AUDIT_MODEL, log=log)
            cache.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

        page_elapsed = time.time() - page_start
        log.info("Page %d/%d done — %.1fs (%d entries)",
                 page_idx, total_pages, page_elapsed, len(data.get("mapping", [])))

        page_texts.append(data.get("sanitized_text", ""))
        page_mappings.append(data.get("mapping", []))
        accumulated_mapping = merge_mapping(accumulated_mapping, data.get("mapping", []), log=log)

    # Step 3 — Build outputs
    log.info("Building outputs (%d unique replacements)", len(accumulated_mapping))
    build_redacted_pdf(stem, page_texts, redacted_path, log=log)
    build_summary_pdf(stem, accumulated_mapping, summary_path, log=log)
    build_governance_json(stem, pdf_path.name, page_mappings,
                          BEDROCK_MODEL, AUDIT_MODEL, governance_path, log=log)

    # Cleanup
    if CLEAN_UP:
        for img_path in image_paths:
            img_path.unlink(missing_ok=True)
        for jf in TEXT_FOLDER.glob(f"{stem}_page_*.json"):
            jf.unlink(missing_ok=True)
        log.info("Temp files cleaned")

    file_elapsed = time.time() - file_start
    log.info("DONE: %s — %.1fs (%d pages, %.1fs/page)",
             pdf_path.name, file_elapsed, total_pages, file_elapsed / total_pages)

    with _progress_lock:
        _completed[0] += 1
        done = _completed[0] + skipped_files
    logger.info("DONE: %s — %.1fs (%d pages) | %d/%d files  [log: logs/%s.log]",
                pdf_path.name, file_elapsed, total_pages, done, total_files, stem)

    return {"stem": stem, "pages": total_pages, "elapsed": file_elapsed, "status": "ok"}


# ── Execute ───────────────────────────────────────────────────
pipeline_start = time.time()
results = []
errors = []

if to_process:
    effective_workers = min(MAX_WORKERS, len(to_process))
    logger.info("Starting ThreadPoolExecutor with %d worker(s)", effective_workers)

    with ThreadPoolExecutor(max_workers=effective_workers) as pool:
        futures = {pool.submit(process_pdf, p): p for p in to_process}
        for future in as_completed(futures):
            pdf_path = futures[future]
            try:
                results.append(future.result())
            except Exception as exc:
                logger.error("FAILED: %s — %s", pdf_path.name, exc)
                errors.append({"stem": pdf_path.stem, "error": str(exc)})

pipeline_elapsed = time.time() - pipeline_start
logger.info("=" * 55)
logger.info("Pipeline complete — %d processed, %d failed, %d skipped, %d total — %.1fs",
            len(results), len(errors), skipped_files, total_files, pipeline_elapsed)
if errors:
    for e in errors:
        logger.error("  FAILED: %s — %s", e["stem"], e["error"])

In [ ]:
# ── Final summary ──────────────────────────────────────────────
for p in sorted(OUTPUT_FOLDER.glob("redacted_*.pdf")):
    logger.info("  %s  (%.1f KB)", p.name, p.stat().st_size / 1024)
for p in sorted(OUTPUT_FOLDER.glob("governance_*.json")):
    logger.info("  %s  (%.1f KB)", p.name, p.stat().st_size / 1024)